# optimizer-loop-on-tensor — ex2: SGD with momentum: extend the per-param loop with a velocity buffer

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `optimizer-loop-on-tensor`. Running the final beacon cell reports progress against the `Optimizer: optimizer.step loop over params` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: optimizer.step loop over params` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-loop-on-tensor`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-loop-on-tensor"
DD_SUBTOPIC = "Optimizer: optimizer.step loop over params"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## SGD with momentum — explicit per-param loop + per-param buffer — quick refresher

Plain SGD's `p -= lr * p.grad` becomes SGD-with-momentum by introducing a per-parameter velocity buffer:
```
for p, b in zip(self.params, self.buffers):
    if p.grad is None:
        continue
    b.copy_(mu * b + p.grad)            # accumulate velocity in place
    p -= self.lr * b                     # step along velocity
```
Three differences from plain SGD:
1. Each parameter gets a `zeros_like(p)` buffer at init.
2. The loop carries one extra `copy_` per param (velocity recurrence).
3. The update reads from `b`, not directly from `p.grad`.

Default momentum `mu = 0.9` matches PyTorch's `torch.optim.SGD` default. The `None`-grad guard and `@t.inference_mode()` decorator carry over unchanged from plain SGD.

### Exercise 2 — SGD with momentum: extend the per-param loop with a velocity buffer

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the per-param explicit loop pattern to SGD-with-momentum, carrying a parallel velocity buffer alongside `self.params` and updating both in place under `@t.inference_mode()`.
> Keywords: sgd-momentum, velocity-buffer, per-param-loop, in-place
> ```

**KCs targeted:** `optimizer-step-explicit-for-loop-over-params`, `per-param-state-buffer-allocation`

Implement `Ex2MomentumSGD`. A hand-rolled SGD with momentum.

1. `__init__(self, params, lr, momentum=0.9)`:
   - Materialize `self.params = list(params)`.
   - Store `self.lr = lr`, `self.momentum = momentum`.
   - Allocate `self.bufs = [t.zeros_like(p) for p in self.params]`.
2. `step(self)` decorated with `@t.inference_mode()`. For each `(p, b)` in `zip(self.params, self.bufs)`:
   - If `p.grad is None`, SKIP.
   - Else: `b.copy_(self.momentum * b + p.grad)`, then `p -= self.lr * b`.
3. `zero_grad(self)`: `p.grad = None` for every param.

The test compares your optimizer to PyTorch's `torch.optim.SGD(..., momentum=0.9)` over 10 steps on the same model + data and asserts the parameter trajectories are identical (within 1e-5 atol). It also verifies the buffer is mutated IN PLACE (same `id` / `data_ptr` across steps).

In [ ]:
class Ex2MomentumSGD:
    def __init__(self, params, lr, momentum=0.9):
        self.params = list(params)
        self.lr = lr
        self.momentum = momentum
        self.bufs = [t.zeros_like(p) for p in self.params]

    @t.inference_mode()
    def step(self):
        for p, b in zip(self.params, self.bufs):
            if p.grad is None:
                continue
            b.copy_(self.momentum * b + p.grad)
            p -= self.lr * b

    def zero_grad(self):
        for p in self.params:
            p.grad = None


<details><summary>Solution</summary>

```python
class Ex2MomentumSGD:
    def __init__(self, params, lr, momentum=0.9):
        self.params = list(params)
        self.lr = lr
        self.momentum = momentum
        self.bufs = [t.zeros_like(p) for p in self.params]

    @t.inference_mode()
    def step(self):
        for p, b in zip(self.params, self.bufs):
            if p.grad is None:
                continue
            b.copy_(self.momentum * b + p.grad)
            p -= self.lr * b

    def zero_grad(self):
        for p in self.params:
            p.grad = None
```

**Why a buffer LIST parallel to params.** Each Parameter has a different shape, so the buffers can't be one big contiguous tensor (without flattening). The parallel-list pattern is what PyTorch's `torch.optim.SGD` does internally too — `state[p]` is a dict keyed by Parameter that stores the velocity buffer.

**Why `b.copy_(...)` not `b = ...`.** Identical issue to EMA-first-moment ex1: rebinding the local loop variable doesn't update the list entry. The next step would read a stale zero buffer and behave like plain SGD with no momentum.

**Match to torch.optim.SGD.** PyTorch's default formula is `v_new = momentum*v + g` (no `(1-mu)` factor — this is the classical-momentum form, not the EMA form). Our impl matches exactly, which is why the trajectories agree to 1e-5 over 10 steps. If you wanted the EMA-style `v_new = momentum*v + (1-momentum)*g` you'd need a different lr scale — that's Adam's first moment, not SGD.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()